In [0]:
from pyspark.sql import functions as F

clean = spark.table("fleetpulse.silver.telemetry_clean")
devices = spark.table("fleetpulse.silver.devices")

# ---- 1. fleet_health_hourly: the general-purpose health mart ----
(clean
  .withColumn("hour", F.date_trunc("hour", F.col("event_ts")))
  .groupBy("device_id", "hour")
  .agg(F.round(F.avg("vibration_mm_s"), 3).alias("avg_vibration"),
       F.round(F.max("vibration_mm_s"), 3).alias("max_vibration"),
       F.round(F.avg("coil_temp_c"), 3).alias("avg_coil_temp"),
       F.round(F.min("helium_pct"), 1).alias("min_helium"),
       F.count("error_code").alias("error_count"),
       F.count("*").alias("event_count"))
  .write.mode("overwrite").saveAsTable("fleetpulse.gold.fleet_health_hourly"))

# ---- 2. device_alerts: statistical outlier detection ----
dev = clean.groupBy("device_id").agg(
    F.avg("vibration_mm_s").alias("avg_vib"),
    F.avg("helium_pct").alias("avg_helium"),
    (F.count("error_code") / F.count("*")).alias("error_rate"))

s = dev.agg(F.expr("percentile(avg_vib, 0.5)").alias("med_vib"),
            F.stddev("avg_vib").alias("sd_vib"),
            F.avg("error_rate").alias("fleet_err")).first()

vib_alerts = (dev.filter(F.col("avg_vib") > s.med_vib + 3 * s.sd_vib)
                 .withColumn("alert_type", F.lit("VIBRATION_ANOMALY")))

# helium: use the device AVERAGE, not the min. Fleet avg ~62.5 with tight
# per-device spread, so avg < 45 fires only for genuinely depleted devices.
hel_alerts = (dev.filter(F.col("avg_helium") < 45)
                 .withColumn("alert_type", F.lit("HELIUM_LOW")))

err_alerts = (dev.filter(F.col("error_rate") > 2 * s.fleet_err)
                 .withColumn("alert_type", F.lit("ERROR_SPIKE")))

alerts = (vib_alerts.unionByName(hel_alerts).unionByName(err_alerts)
          .join(devices, "device_id", "left"))
alerts.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable("fleetpulse.gold.device_alerts")

# ---- 3. sla_daily: reporting rate per hospital ----
(clean
  .withColumn("day", F.to_date("event_ts"))
  .join(devices, "device_id")
  .groupBy("hospital", "day")
  .agg(F.countDistinct("device_id").alias("devices_reporting"),
       F.count("*").alias("events_received"))
  .join(devices.groupBy("hospital").agg(F.countDistinct("device_id").alias("devices_total")), "hospital")
  .withColumn("reporting_pct", F.round(100 * F.col("devices_reporting") / F.col("devices_total"), 1))
  .write.mode("overwrite").saveAsTable("fleetpulse.gold.sla_daily"))

In [0]:
%sql
SELECT device_id, alert_type, ROUND(avg_vib, 2) AS avg_vib, hospital
FROM fleetpulse.gold.device_alerts
WHERE alert_type = 'VIBRATION_ANOMALY'
ORDER BY device_id;

In [0]:
%sql
SELECT alert_type, COUNT(*) FROM fleetpulse.gold.device_alerts GROUP BY alert_type